In [ ]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError
from lxml_html_clean import clean_html

import csv
import logging
import time
from multiprocessing.pool import ThreadPool
from sys import stdin, stdout
from time import perf_counter
from typing import Iterator, List, Optional

import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher, PubMedArticle
from metapub import pubmedcentral


from sqlitedict import SqliteDict

In [ ]:
pmid_clean_list = np.load('PMID_lists/pmids_'+'.npy')

In [ ]:
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type

retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([EutilsNCBIError, EutilsRequestError])
)()

In [ ]:

article_cache = SqliteDict("pubmed_cache.db", "articles")
fetcher = PubMedFetcher()


def fetch_article(pmid: str) -> PubMedArticle:
    "call fetcher.article_by_pmid with lots of logging statements"
    # log as this is going to take a long time
    logging.info("fetching article with pmid=%r", pmid)

    t0 = perf_counter()
    article = fetcher.article_by_pmid(pmid)
    dt = perf_counter() - t0
    logging.info("fetched pmid=%s in %.3fms", pmid, dt * 1000)

    # check if we got the right thing back
    if article.pmid != pmid:
        logging.warning("article with pmid=%r returned pmid=%r", pmid, article.pmid)

    return article


def fetch_article_cached(pmid: str) -> PubMedArticle:
    """cache XML of fetched articles so subsequent runs are faster.
    errors are returned as a new PubMedArticle object with an empty XML string
    """
    try:
        xml = article_cache[pmid]
    except KeyError:
        # cached failed, continue below with fetching
        pass
    else:
        # cache hit, parse it
        return PubMedArticle(xml)

    try:
        article = fetch_article(pmid)
    except Exception as err:
        logging.warning("error fetching pmid=%r: %s", pmid, err)
        article = PubMedArticle("")  # Create a new PubMedArticle with empty XML
    # add a delay to avoid exceeding the API rate limit
    time.sleep(1 / 3)

    # save xml in cache for next time
    if article is not None:
        article_cache[pmid] = article.xml

    return article


@retry_on_communication_error
def fetch_articles(
    pmids: List[str], *, processes: Optional[int] = None
) -> Iterator[PubMedArticle]:
    "use a threadpool to fetch articles from metapub in parallel, caching where possible"
    with ThreadPool(processes=processes) as pool:
        for article in pool.imap_unordered(fetch_article_cached, pmids):
            # couldn't be fetched
            if article == None:
                continue
            yield article


def main() -> None:
    "fetch articles by PMID from a list called pmid_clean_list and write to a CSV file"
    with open("/article_metadata/output.csv", "w", newline="", encoding="utf-8") as f:
        out = csv.writer(f)
        first = True
        for article in fetch_articles(pmid_clean_list, processes=6):
            row = dict(
                pmid=article.pmid,
                pmc=article.pmc,
                doi=article.doi,
                date=article.history,
                title=article.title,
                journal=article.journal,
                abstract=article.abstract,
                chemicals=[v["substance_name"] for v in article.chemicals.values()],
                mesh_descriptors=[v["descriptor_name"] for v in article.mesh.values()],
                mesh_qualifiers=[v["qualifier_name"] for v in article.mesh.values()],
                url=article.url,
            )
            if first:
                out.writerow(row.keys())
                first = False
            out.writerow(row.values())

if __name__ == "__main__":  #Cleaning up the test database to keep each doctest run idempotent:
    logging.basicConfig(level=logging.DEBUG)
    main()
